In [ ]:
from typing import Annotated, Final, Tuple, Literal, TypedDict
from langgraph.checkpoint.postgres import PostgresSaver
from langgraph.store.postgres import PostgresStore
from loadmodel import load_model, load_postgresconfig
from langgraph.graph.message import MessagesState
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import StateGraph, START, END
from langchain.messages import HumanMessage,SystemMessage,AIMessage,ToolMessage
from langgraph import graph
from langgraph.runtime import Runtime
from dataclasses import dataclass
from loguru import logger

DB_DSN = load_postgresconfig()
# 初始化模型
model = load_model()


# 1. 定义运行时的环境上下文
@dataclass
class UserContext:
    username: str
    membership_level: str

# 2. 定义状态
class OverAllState(MessagesState):
    user_input: str
    output: str

# 3. 定义节点
def llm_node(state:OverAllState, runtime:Runtime[UserContext]) -> OverAllState:
    # 1. 获取环境上下文，判断当前用户等级
    runtime_context = runtime.context
    
    if runtime_context:
        level = runtime_context.membership_level
        username = runtime_context.username
        logger.info(f"当前用户等级：{level}，用户名：{username}")
        if level == "VIP":
            system_prompt = f"你是高级客户助理，当前VIP用户是{username},请在回复的时候语气热情周到，回复末尾加上'这里是VIP服务，感谢您的支持'"
        else:
            system_prompt = "你是普通客户助理，普通用户没有特殊要求，友好简洁的回答用户的问题即可"
    else:
        system_prompt = "你是普通客户助理，普通用户没有特殊要求，友好简洁的回答用户的问题即可"

    user_input = state["user_input"]
    messages = state.get("messages", [])
    response = model.invoke([SystemMessage(content=system_prompt)]+messages+[HumanMessage(content=user_input)])
    
    return {
        "messages": messages,
        "output": response.content
    }


# 4. 定义图
builder = StateGraph(state_schema=OverAllState, context_schema=UserContext)

builder.add_node("llm_node", llm_node)
builder.add_edge(START, "llm_node")
builder.add_edge("llm_node", END)

graph = builder.compile()

#  第一次调用：传入VIP上下文
result = graph.invoke(
    {"user_input": "你好，帮我查一下有什么活动？"},
    context = UserContext(username="Alice", membership_level="VIP")
)
print(result)



2026-08-20 15:04:17.740 | INFO     | __main__:llm_node:38 - 当前用户等级：VIP，用户名：Alice


{'messages': [], 'user_input': '你好，帮我查一下有什么活动？', 'output': '亲爱的 Alice，您好呀！👋 非常高兴能为您服务！\n\n为了给您提供最精准、最贴心的活动推荐，我想先了解一下您目前比较感兴趣的方向呢？比如：\n\n*   🎁 **专属礼遇**：查看针对 VIP 用户的积分倍增、生日特权或定制礼品；\n*   ✈️ **出行优惠**：查询近期的机票折扣、酒店升级或旅行套餐；\n*   🛍️ **购物促销**：了解最新的名品折扣、限时秒杀或新品首发活动；\n*   🎉 **线下体验**：咨询高端品鉴会、私人沙龙或会员专属派对。\n\n您可以告诉我您的偏好，或者如果您有特定的时间计划，我也可以为您量身定制一份活动清单哦！期待您的回复，我会立刻为您详细查询！✨\n\n这里是VIP服务，感谢您的支持'}


In [3]:
#  第二次调用：传入普通用户上下文
result1 = graph.invoke(
    {"user_input": "你好，帮我查一下有什么活动？"},
    context = UserContext(username="Alice", membership_level="普通")
)
print(result1)

2026-08-20 15:05:30.146 | INFO     | __main__:llm_node:38 - 当前用户等级：普通，用户名：Alice


{'messages': [], 'user_input': '你好，帮我查一下有什么活动？', 'output': '您好！很高兴为您服务。\n\n由于我不确定您具体指的是哪个平台或品牌（例如：某家银行、电商平台、运营商或具体的APP），暂时无法直接为您列出确切的活动清单。\n\n您可以告诉我**具体的平台名称**或**业务类型**吗？或者，您也可以直接登录对应的官方APP或网站，通常在首页的“活动中心”、“热门优惠”或“公告栏”中能看到最新的实时活动信息。\n\n如果有具体目标，欢迎补充，我会尽力为您提供更详细的指引！'}


# 总结
## 适合放入运行时的环境上下文
- 当前登录用户信息
- 请求来源（如：web、app、api等）
- 调用方信息（如：ip地址、用户代理等）
- 本次调用的功能开关
## 适合放入图状态的上下文
- 需要在多轮对话间共享的数据
- 需要在检查点钟恢复的执行进度
- 需要跨调用持久化的业务数据
- 节点间需要传递的计算结果